In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

import os

load_dotenv()

llm = ChatOpenAI(
    base_url = os.getenv("OPENAI_BASE_URL"),
    api_key = os.getenv("OPENAI_API_KEY"), 
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
    temperature = 0.1,
)

In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.memory import ConversationBufferMemory, ConversationSummaryBufferMemory, ConversationBufferWindowMemory 

buffer_memory = ConversationBufferMemory(return_messages=True)

buffer_window_memory = ConversationBufferWindowMemory(k=3, return_messages=True)

summary_memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=100,
    return_messages=True
)

conversations = [
    ("Hi, I'm Nicolas and I live in South Korea.", "Nice to meet you, Nicolas from South Korea."),
    ("I work as a software engineer.", "Got it. You work as a software engineer."),
    ("My favorite food is kimchi.", "Understood. Your favorite food is kimchi."),
    ("I have a dog named Choco.", "Okay. Your dog is named Choco."),
    ("I usually wake up at 6 AM.", "Noted. You usually wake up at 6 AM."),
    ("On weekends I like hiking.", "Understood. You like hiking on weekends."),
    ("I'm learning Spanish these days.", "Got it. You are learning Spanish these days."),
    ("My favorite soccer team is Tottenham.", "Okay. Your favorite soccer team is Tottenham."),
    ("I want to visit Japan next year.", "Noted. You want to visit Japan next year."),
    ("I prefer coffee over tea.", "Got it. You prefer coffee over tea."),
]

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", ""),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

for user_input, ai_response in conversations:
    buffer_memory.save_context({"input": user_input}, {"output": ai_response})
    buffer_window_memory.save_context({"input": user_input}, {"output": ai_response})
    summary_memory.save_context({"input": user_input}, {"output": ai_response})

def load_buffer_memory(_):
    return buffer_memory.load_memory_variables({})["history"]

buffer_chain = RunnablePassthrough.assign(history=load_buffer_memory) | prompt | llm 


print("=== Buffer Memory ===")
print(buffer_chain.invoke({
    "input": "What do you know about me?"
}).content)
print("Check 1st conversation:")
print(buffer_chain.invoke({
    "input": "What is my name?"
}).content)
print("Memory token count:", llm.get_num_tokens_from_messages(buffer_memory.load_memory_variables({})["history"]))

=== Buffer Memory ===
Here’s what I know about you so far, Nicolas:

- You live in South Korea.  
- You work as a software engineer.  
- Your favorite food is kimchi.  
- You have a dog named Choco.  
- You usually wake up at 6 AM.  
- On weekends, you like hiking.  
- You’re learning Spanish these days.  
- Your favorite soccer team is Tottenham.  
- You want to visit Japan next year.  
- You prefer coffee over tea.

If you’d like, I can use these details to personalize examples (e.g., coding tasks, Spanish practice, travel plans to Japan, or hiking/fitness schedules).
Check 1st conversation:
Your name is Nicolas.
Memory token count: 257


In [5]:
def load_buffer_window_memory(_):
    return buffer_window_memory.load_memory_variables({})["history"]

window_chain = RunnablePassthrough.assign(history=load_buffer_window_memory) | prompt | llm 

print("=== Buffer Window Memory ===")
print(window_chain.invoke({
    "input": "What do you know about me?"
}).content)
print("Check 1st conversation:")
print(window_chain.invoke({
    "input": "What is my name?"
}).content)
print("Memory token count:", llm.get_num_tokens_from_messages(buffer_window_memory.load_memory_variables({})["history"]))


=== Buffer Window Memory ===
Here’s what you’ve told me so far:

1. Your favorite soccer team is Tottenham Hotspur.  
2. You want to visit Japan next year.  
3. You prefer coffee over tea.

I don’t know anything about you beyond what you’ve explicitly shared in this conversation. I also don’t retain this information once the session ends unless the system you’re using adds its own memory layer.
Check 1st conversation:
You haven’t told me your name yet, so I don’t know it.
Memory token count: 77


In [6]:
def load_summary_memory(_):
    return summary_memory.load_memory_variables({})["history"]

summary_chain = RunnablePassthrough.assign(history=load_summary_memory) | prompt | llm

print("=== Summary Memory ===")
print(summary_chain.invoke({
    "input": "What do you know about me?"
}).content)
print("Check 1st conversation:")
print(summary_chain.invoke({
    "input": "What is my name?"
}).content)
print("Memory token count:", llm.get_num_tokens_from_messages(summary_memory.load_memory_variables({})["history"]))

=== Summary Memory ===
Here’s what I know about you so far, Nicolas:

- You live in South Korea.  
- You work as a software engineer.  
- Your favorite food is kimchi.  
- You have a dog named Choco.  
- You usually wake up at 6 AM.  
- On weekends, you like hiking.  
- You’re learning Spanish these days.  
- Your favorite soccer team is Tottenham.  
- You want to visit Japan next year.  
- You prefer coffee over tea.
Check 1st conversation:
Your name is Nicolas.
Memory token count: 198
